## Q1. Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

 Ans -- In a Spark application, the Driver is the main program that controls everything. It creates the Spark session, divides the work into small tasks, and sends them to the executors. The Cluster Manager is responsible for managing the resources of the cluster, such as CPU and memory, and assigning them to the application. The Executors run on worker nodes and perform the tasks given by the Driver. After completing the tasks, they send the results back to the Driver.

## Q2. How does Spark's Lazy Evaluation strategy improve performance?

Ans -- Spark uses Lazy Evaluation, which means it does not execute transformations immediately. Instead, it remembers all the operations and creates a plan called a DAG (Directed Acyclic Graph). When an action like show() or count() is called, Spark executes the plan in the most efficient way. This reduces unnecessary work and improves performance.

## Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled. 

In [0]:
df = spark.read.csv(
    "/Volumes/sample_data/default/source/source.csv",
    header=True,
    inferSchema=True
)

In [0]:
df.show()

+--------+----------+--------+-----------+-------+----------+------+--------+---------+-------+-------+
|order_id|product_id|old_name|   category|  price|base_price|region|priority|   status| amount|user_id|
+--------+----------+--------+-----------+-------+----------+------+--------+---------+-------+-------+
| ORD2001|      P101|    BOOK|      Books| 566.81|    566.81| North|    High|Cancelled|1700.43|  U1001|
| ORD2002|      P102|    CLTH|   Clothing|1426.47|   1426.47| North|     Low|Cancelled|7132.35|   NULL|
| ORD2003|      P103|    GROC|    Grocery|1100.13|   1100.13| North|    High|Completed|2200.26|  U1003|
| ORD2004|      P104|    SPRT|     Sports|  901.2|     901.2| North|     Low|Completed| 4506.0|  U1004|
| ORD2005|      P105|    GROC|    Grocery| 240.78|    240.78|  East|    High|Completed| 963.12|  U1005|
| ORD2006|      P106|    CLTH|   Clothing| 240.74|    240.74| South|  Medium|Completed| 240.74|  U1006|
| ORD2007|      P107|    FURN|  Furniture|  94.66|     94.66|  E

# Q4. What is the difference between CSV and Parquet in terms of storage and performance?

Ans -- CSV is a row-based file format, while Parquet is a column-based file format. CSV files are simple to read but usually take more storage space and are slower to process. Parquet files are compressed, use less storage, and are much faster because Spark reads only the required columns. This makes Parquet a better choice for big data processing.

In [0]:
df.filter(df.category == "Electronics").select("product_id", "price").show()

+----------+-------+
|product_id|  price|
+----------+-------+
|      P115| 279.28|
|      P120| 442.51|
|      P129| 891.88|
|      P135|1448.72|
|      P139|1028.88|
|      P140| 664.71|
|      P142|  746.8|
|      P147| 473.07|
|      P149| 823.69|
|      P153|1409.73|
|      P154|1343.08|
|      P155| 900.07|
|      P158| 300.41|
+----------+-------+



# Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'? 

#  Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double. 

In [0]:
from pyspark.sql.functions import col

df = df.withColumnRenamed("old_name", "new_name")
df = df.withColumn("price", col("price").cast("double"))

In [0]:
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- new_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- user_id: string (nullable = true)



# Q7. How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

Ans -- Spark keeps a record of all the transformations performed on the data using a Lineage Graph (DAG). If a worker node fails, Spark does not need to restart the entire job. Instead, it uses the DAG to recompute only the lost data. This makes Spark reliable and fault tolerant.

# Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000. 

In [0]:
df.filter(
    (df.status == "Completed") &
    (df.amount > 1000)
).show()

+--------+----------+--------+-----------+-------+----------+------+--------+---------+-------+-------+
|order_id|product_id|new_name|   category|  price|base_price|region|priority|   status| amount|user_id|
+--------+----------+--------+-----------+-------+----------+------+--------+---------+-------+-------+
| ORD2003|      P103|    GROC|    Grocery|1100.13|   1100.13| North|    High|Completed|2200.26|  U1003|
| ORD2004|      P104|    SPRT|     Sports|  901.2|     901.2| North|     Low|Completed| 4506.0|  U1004|
| ORD2008|      P108|    SPRT|     Sports|1300.33|   1300.33| North|  Medium|Completed|6501.65|  U1008|
| ORD2010|      P110|    CLTH|   Clothing|1064.44|   1064.44|  East|    High|Completed|1064.44|  U1010|
| ORD2019|      P119|    GROC|    Grocery| 652.46|    652.46|  East|    High|Completed| 3262.3|  U1019|
| ORD2028|      P128|    FURN|  Furniture| 775.24|    775.24| South|    High|Completed| 3876.2|  U1028|
| ORD2034|      P134|    SPRT|     Sports|1423.74|   1423.74| So

# Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory. 

Ans -- Predicate Pushdown is an optimization technique in which filter conditions are pushed down to the data source while reading the file. In Parquet files, Spark reads only the required rows instead of scanning the entire dataset. This reduces disk I/O and improves query performance.

# Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax). 

In [0]:
from pyspark.sql.functions import col

df = df.withColumn("final_price", col("base_price") * 1.18)

+--------+----------+--------+-----------+-------+----------+------+--------+---------+-------+-------+------------------+
|order_id|product_id|new_name|   category|  price|base_price|region|priority|   status| amount|user_id|       final_price|
+--------+----------+--------+-----------+-------+----------+------+--------+---------+-------+-------+------------------+
| ORD2001|      P101|    BOOK|      Books| 566.81|    566.81| North|    High|Cancelled|1700.43|  U1001|          668.8358|
| ORD2002|      P102|    CLTH|   Clothing|1426.47|   1426.47| North|     Low|Cancelled|7132.35|   NULL|         1683.2346|
| ORD2003|      P103|    GROC|    Grocery|1100.13|   1100.13| North|    High|Completed|2200.26|  U1003|1298.1534000000001|
| ORD2004|      P104|    SPRT|     Sports|  901.2|     901.2| North|     Low|Completed| 4506.0|  U1004|          1063.416|
| ORD2005|      P105|    GROC|    Grocery| 240.78|    240.78|  East|    High|Completed| 963.12|  U1005|284.12039999999996|
| ORD2006|      

# Q11: What is the difference between Transformations and Actions? Provide two examples of each. 
Ans -- Transformations are operations that create a new DataFrame but do not execute immediately. They are executed only when an action is called. Examples of transformations are filter() and select(). Actions are operations that trigger the execution of the transformations and return the result. Examples of actions are show() and count().

# Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output". 

In [0]:
df = spark.read.parquet(
    "/Volumes/sample_data/default/source/source.parquet"
)

df.filter(df.user_id.isNotNull()) \
  .write \
  .mode("overwrite") \
  .option("header", True) \
  .csv("/Volumes/sample_data/default/source/output_csv")

# Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode? 
Ans -- In Client Mode, the Driver program runs on the user's machine, while the executors run on the cluster. This mode is mainly used for development and testing. In Cluster Mode, the Driver also runs inside the cluster. This is better for production because the application continues running even if the user disconnects.

# Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'?

In [0]:
df.filter((df.region == "North") | (df.priority == "High")).show()

+--------+----------+--------+-----------+-------+----------+------+--------+---------+-------+-------+
|order_id|product_id|old_name|   category|  price|base_price|region|priority|   status| amount|user_id|
+--------+----------+--------+-----------+-------+----------+------+--------+---------+-------+-------+
| ORD2001|      P101|    BOOK|      Books| 566.81|    566.81| North|    High|Cancelled|1700.43|  U1001|
| ORD2002|      P102|    CLTH|   Clothing|1426.47|   1426.47| North|     Low|Cancelled|7132.35|   NULL|
| ORD2003|      P103|    GROC|    Grocery|1100.13|   1100.13| North|    High|Completed|2200.26|  U1003|
| ORD2004|      P104|    SPRT|     Sports|  901.2|     901.2| North|     Low|Completed| 4506.0|  U1004|
| ORD2005|      P105|    GROC|    Grocery| 240.78|    240.78|  East|    High|Completed| 963.12|  U1005|
| ORD2008|      P108|    SPRT|     Sports|1300.33|   1300.33| North|  Medium|Completed|6501.65|  U1008|
| ORD2010|      P110|    CLTH|   Clothing|1064.44|   1064.44|  E

# Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset? 
Ans -- The show(5) method displays only the first five rows of the DataFrame, so it uses very little memory and is safe for large datasets. The collect() method brings all the data to the Driver machine, which can use a lot of memory and may even cause the application to fail if the dataset is very large. Therefore, show(5) is a better choice for exploring large datasets.